# 06 -- Farmer movement, second increment: expand the loop to 12 tiles

`notebooks/05_farmer_movement.ipynb` validated a 2x2 tile loop
(`wheat_loop_2x2_agent`) that beats the single-tile baseline. This
notebook pushes the same idea further: how big can the loop get before
diminishing returns (or a hard turn-budget ceiling) kick in, and does a
bigger loop actually keep winning against the *current* best agent, not
just the original baseline?

## Picking a target size: the ~12 tiles/day ceiling

`turnsPerDay = 24`, and moving costs the same one turn as tending a
tile. Visiting a new tile costs roughly 1 move + 1 tend action, so a
single farmer can service at most about `24 / 2 ~= 12` tiles per day.
Rather than grow the loop by trial and error, this increment targets
that ceiling directly: 12 tiles.

In [1]:
import sys
from pathlib import Path

from kaggle_environments import make

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggriculture_bot.evaluate import evaluate
from kaggriculture_bot.strategies.baseline import wheat_loop_2x2_agent


## Building a 12-tile closed loop

Same board as before: farmer spawns at `(4,4)`, the NW quadrant is
`x,y in [0,4]`. Construction: start from the 2-row rectangle spanning
`y=3..4` across all of `x=0..4` -- that alone is already a closed loop
of 10 tiles (a 2-row rectangle has no interior, so every cell is on its
own boundary). To reach 12 without revisiting any tile, insert a 2-tile
detour into row `y=2` along one stretch of the rectangle, swapping a
single direct step for a 4-step path through 2 new, previously-unvisited
cells:

```
(4,4) -> (3,4) -> (2,4) -> (1,4) -> (0,4)   # bottom row, right to left
      -> (0,3) -> (1,3) -> (2,3)             # up, then left to right
      -> (2,2) -> (3,2)                      # detour up into y=2
      -> (3,3) -> (4,3)                      # back down, close the row
      -> (4,4)                               # back to spawn
```

Every consecutive pair is orthogonally adjacent (checked below, same
test as the 2x2 loop's), all 12 tiles are distinct, and every direction
used is one of the 4 already in `DIRECTIONS` -- no new entries needed
there.

In [2]:
LOOP_ORDER_12 = {
    (4, 4): (3, 4),
    (3, 4): (2, 4),
    (2, 4): (1, 4),
    (1, 4): (0, 4),
    (0, 4): (0, 3),
    (0, 3): (1, 3),
    (1, 3): (2, 3),
    (2, 3): (2, 2),
    (2, 2): (3, 2),
    (3, 2): (3, 3),
    (3, 3): (4, 3),
    (4, 3): (4, 4),
}
DIRECTIONS = {(0, -1): "NORTH", (0, 1): "SOUTH", (1, 0): "EAST", (-1, 0): "WEST"}


def wheat_loop_12_agent(obs):
    market = []
    wheat_seeds = obs["private"]["seeds"].get("WHEAT", 0)
    if obs["step"] == 0 or wheat_seeds <= 0:
        market.append(["BUY_SEED", "WHEAT", 1])

    farmer_pos = obs["farms"][obs["player"]]["farmer"]
    x, y = farmer_pos
    target_x, target_y = LOOP_ORDER_12[(x, y)]
    direction = DIRECTIONS[(target_x - x, target_y - y)]

    tile = obs["farms"][obs["player"]]["tiles"][y][x]

    wheat_in_shed = obs["private"]["shed"].get("WHEAT", 0)
    if wheat_in_shed > 0:
        market.append(["SELL", "WHEAT", 1])

    if tile is None and wheat_seeds > 0:
        farmer = ["PLANT", "WHEAT"]
    elif tile is not None and tile.get("kind") == "PLANT":
        crop_age = obs["day"] - tile["planted_day"]
        if crop_age >= 2:
            farmer = ["HARVEST"]
        elif not tile.get("watered_today"):
            farmer = ["WATER"]
        else:
            farmer = [direction]
    else:
        farmer = [direction]

    return {"farmer": farmer, "hands": [], "market": market}


## Sanity check: is the loop actually a valid closed cycle?

Before running a 720-turn episode, verify the hand-built loop mechanically:
every step lands on an orthogonally adjacent tile, all 12 tiles are
distinct, and it closes back to the start after exactly 12 steps. (This
is the same check that graduated into
`tests/test_baseline.py::assert_is_closed_adjacent_cycle`.)

In [3]:
def assert_is_closed_adjacent_cycle(loop_order):
    n = len(loop_order)
    start = next(iter(loop_order))
    pos = start
    visited = set()
    for _ in range(n):
        visited.add(pos)
        (x, y), (nx, ny) = pos, loop_order[pos]
        assert abs(nx - x) + abs(ny - y) == 1
        pos = (nx, ny)
    assert pos == start
    assert visited == set(loop_order)


assert_is_closed_adjacent_cycle(LOOP_ORDER_12)
print("LOOP_ORDER_12 is a valid 12-tile closed loop")


LOOP_ORDER_12 is a valid 12-tile closed loop


## Validating against the current best, not the original baseline

Per this project's rule (`structuring-ml-projects` step 4: recompute
the baseline in the same run, and the gate compares against the current
state, not a stale one): the agent to beat here is `wheat_loop_2x2_agent`,
the best graduated so far -- not the original single-tile
`wheat_loop_agent`.

In [4]:
print(evaluate(wheat_loop_2x2_agent, wheat_loop_12_agent, n=20))


{'win_rate': 0.0, 'avg_money': 4333.0}


As with the 2x2 vs. 1-tile comparison, a single `evaluate()` call's `n`
episodes aren't independent samples (the reused `env` object doesn't
reseed between `.run()` calls -- see `notebooks/05_farmer_movement.ipynb`
and the 2026-08-19 harness note in `PROGRESS.md`). Confirming the result
holds across separate calls:

In [5]:
print(evaluate(wheat_loop_2x2_agent, wheat_loop_12_agent, n=20))


{'win_rate': 0.0, 'avg_money': 4403.0}


### Conclusion

`wheat_loop_2x2_agent` loses 20/20 to `wheat_loop_12_agent` in both
calls, with `avg_money` 4301.0 and 4413.0 for the 2x2 loop (the losing
side) -- consistent with the ~4293 it scored against the 1-tile baseline
in the previous notebook, and clearly beaten by the 12-tile version both
times.

Graduated to
`src/kaggriculture_bot/strategies/baseline.py::wheat_loop_12_agent`, with
unit tests in `tests/test_baseline.py` (loop structure, plus behavior on
both the plain stretch and the 2-tile detour specifically) and a
`RESOURCES.md` entry. Not yet wired into `agent/main.py` -- see
`PROGRESS.md`'s Next steps for that decision and the still-open day-0
startup-gap question.